In [2]:
from pathlib import Path
import sys
import pandas as pd
import pickle
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

from src.attention_model import AttentionDecoderLSTM, generate_caption_attention
from src.vocab import Vocab, tokenize, PAD, SOS, EOS, UNK

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
train_df = pd.read_csv(ARTIFACTS_DIR / "train.csv")
val_df   = pd.read_csv(ARTIFACTS_DIR / "val.csv")
test_df  = pd.read_csv(ARTIFACTS_DIR / "test.csv")

with open(ARTIFACTS_DIR / "vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

with open(ARTIFACTS_DIR / "features_spatial.pkl", "rb") as f:
    feat_data = pickle.load(f)

filenames_list = feat_data["filenames"]
feature_matrix = feat_data["matrix"]  # (8091, 49, 2048)
fname_to_idx = {f: i for i, f in enumerate(filenames_list)}

print(train_df.shape, val_df.shape, test_df.shape, len(vocab.itos), feature_matrix.shape)

(32360, 2) (4045, 2) (4050, 2) 2652 torch.Size([8091, 49, 2048])


In [4]:
class CaptionDatasetAttention(Dataset):
    def __init__(self, df, feature_matrix, fname_to_idx, vocab):
        self.df = df.reset_index(drop=True)
        self.feature_matrix = feature_matrix
        self.fname_to_idx = fname_to_idx
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row["image"]
        caption = row["caption"]

        feat_idx = self.fname_to_idx[fname]
        feature = self.feature_matrix[feat_idx].float()  # (49, 2048)

        tokens = tokenize(caption)
        ids = self.vocab.encode(tokens)

        return feature, torch.tensor(ids, dtype=torch.long)


def collate_fn(batch):
    features, captions = zip(*batch)
    features = torch.stack(features)  # (batch, 49, 2048)
    lengths = torch.tensor([len(c) for c in captions])
    captions_padded = pad_sequence(captions, batch_first=True, padding_value=vocab.stoi[PAD])
    return features, captions_padded, lengths


train_dataset = CaptionDatasetAttention(train_df, feature_matrix, fname_to_idx, vocab)
val_dataset   = CaptionDatasetAttention(val_df, feature_matrix, fname_to_idx, vocab)
test_dataset  = CaptionDatasetAttention(test_df, feature_matrix, fname_to_idx, vocab)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

feat_batch, cap_batch, len_batch = next(iter(train_loader))
print(feat_batch.shape, cap_batch.shape)  # expect (32, 49, 2048), (32, seq_len)

torch.Size([32, 49, 2048]) torch.Size([32, 22])


In [5]:
model = AttentionDecoderLSTM(
    feature_dim=2048,
    embed_dim=256,
    hidden_dim=512,
    vocab_size=len(vocab.itos),
    attn_dim=256,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi[PAD])
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# sanity check: untrained loss should be close to ln(vocab_size), same
# check that caught real bugs in the baseline model last time
import math
print("expected ~random loss:", math.log(len(vocab.itos)))

feat_batch = feat_batch.to(device)
cap_batch = cap_batch.to(device)
inputs = cap_batch[:, :-1]
targets = cap_batch[:, 1:]
logits = model(feat_batch, inputs)
loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
print("actual first-batch loss:", loss.item())

expected ~random loss: 7.883069351305753
actual first-batch loss: 7.8732500076293945


In [6]:
def train_one_batch(features, captions):
    features = features.to(device)
    captions = captions.to(device)

    inputs = captions[:, :-1]
    targets = captions[:, 1:]

    optimizer.zero_grad()
    logits = model(features, inputs)
    loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()

    return loss.item()


def evaluate(loader):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for features, captions, lengths in loader:
            features = features.to(device)
            captions = captions.to(device)
            inputs = captions[:, :-1]
            targets = captions[:, 1:]
            logits = model(features, inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            total_loss += loss.item()
            n_batches += 1
    model.train()
    return total_loss / n_batches


def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for features, captions, lengths in tqdm(loader):
        loss = train_one_batch(features, captions)
        total_loss += loss
        n_batches += 1
    return total_loss / n_batches

In [7]:
# Same checkpointing lesson as the baseline: save only when val loss
# improves, so a later overfit epoch never overwrites the best model.
NUM_EPOCHS = 15
best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(train_loader)
    val_loss = evaluate(val_loader)
    print(f"epoch {epoch+1}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), ARTIFACTS_DIR / "best_model_attention.pt")
        print(f"  -> saved new best attention model (val_loss={val_loss:.4f})")

100%|██████████| 1012/1012 [02:19<00:00,  7.24it/s]


epoch 1/15  train_loss=3.3011  val_loss=2.8590
  -> saved new best attention model (val_loss=2.8590)


100%|██████████| 1012/1012 [02:20<00:00,  7.23it/s]


epoch 2/15  train_loss=2.4838  val_loss=2.7310
  -> saved new best attention model (val_loss=2.7310)


100%|██████████| 1012/1012 [02:18<00:00,  7.29it/s]


...

epoch 4/15  train_loss=1.7969  val_loss=2.7762


100%|██████████| 1012/1012 [02:18<00:00,  7.31it/s]


epoch 5/15  train_loss=1.5261  val_loss=2.8528


 61%|██████    | 619/1012 [01:29<00:56,  6.91it/s]


KeyboardInterrupt: 

In [8]:
inference_model = AttentionDecoderLSTM(
    feature_dim=2048, embed_dim=256, hidden_dim=512, vocab_size=len(vocab.itos), attn_dim=256
)
inference_model.load_state_dict(torch.load(ARTIFACTS_DIR / "best_model_attention.pt"))
inference_model = inference_model.to(device)
inference_model.eval()
print("loaded")

loaded


In [9]:
# Same repetition-loop dog image from the baseline/beam-search testing --
# the single most useful comparison case we have.
dog_row = test_df[test_df["image"] == "3259002340_707ce96858.jpg"].iloc[0]
dog_feat_idx = fname_to_idx[dog_row["image"]]
dog_feature = feature_matrix[dog_feat_idx].float()

caption, alphas = generate_caption_attention(dog_feature, inference_model, vocab)
print("Attention model:", caption)
print("Ground truth:", dog_row["caption"])

Attention model: a black dog with a blue collar and a blue collar is running in the grass
Ground truth: A black dog with a purple collar and a black leash runs in the grass .


In [10]:
import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method4

def compute_bleu_attention(test_df, feature_matrix, fname_to_idx, model, vocab):
    refs_by_image = test_df.groupby("image")["caption"].apply(list).to_dict()
    bleu1_scores, bleu4_scores = [], []

    for fname, references in refs_by_image.items():
        feat_idx = fname_to_idx[fname]
        feature = feature_matrix[feat_idx].float()

        generated, _ = generate_caption_attention(feature, model, vocab)
        gen_tokens = generated.split()
        ref_tokens_list = [tokenize(ref) for ref in references]

        bleu1 = sentence_bleu(ref_tokens_list, gen_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothie)
        bleu4 = sentence_bleu(ref_tokens_list, gen_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
        bleu1_scores.append(bleu1)
        bleu4_scores.append(bleu4)

    return sum(bleu1_scores) / len(bleu1_scores), sum(bleu4_scores) / len(bleu4_scores)


avg_bleu1, avg_bleu4 = compute_bleu_attention(test_df, feature_matrix, fname_to_idx, inference_model, vocab)
print(f"Attention model BLEU-1: {avg_bleu1:.4f}  (baseline was 0.5693)")
print(f"Attention model BLEU-4: {avg_bleu4:.4f}  (baseline was 0.1709)")

Attention model BLEU-1: 0.5738  (baseline was 0.5693)
Attention model BLEU-4: 0.1734  (baseline was 0.1709)


In [14]:
from importlib import reload
import src.attention_model as attention_model_module
reload(attention_model_module)
from src.attention_model import beam_search_caption_attention

# same repetition-loop dog image
caption = beam_search_caption_attention(dog_feature, inference_model, vocab, beam_width=8)
print("Attention + Beam-8:", caption)
print("Ground truth:", dog_row["caption"])

Attention + Beam-8: a black dog with a toy in his mouth
Ground truth: A black dog with a purple collar and a black leash runs in the grass .


In [12]:
def compute_bleu_attention_beam(test_df, feature_matrix, fname_to_idx, model, vocab, beam_width=8):
    refs_by_image = test_df.groupby("image")["caption"].apply(list).to_dict()
    bleu1_scores, bleu4_scores = [], []

    for fname, references in refs_by_image.items():
        feat_idx = fname_to_idx[fname]
        feature = feature_matrix[feat_idx].float()

        generated = beam_search_caption_attention(feature, model, vocab, beam_width=beam_width)
        gen_tokens = generated.split()
        ref_tokens_list = [tokenize(ref) for ref in references]

        bleu1 = sentence_bleu(ref_tokens_list, gen_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothie)
        bleu4 = sentence_bleu(ref_tokens_list, gen_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
        bleu1_scores.append(bleu1)
        bleu4_scores.append(bleu4)

    return sum(bleu1_scores) / len(bleu1_scores), sum(bleu4_scores) / len(bleu4_scores)


avg_bleu1, avg_bleu4 = compute_bleu_attention_beam(test_df, feature_matrix, fname_to_idx, inference_model, vocab, beam_width=8)
print(f"Attention + Beam-8 BLEU-1: {avg_bleu1:.4f}")
print(f"Attention + Beam-8 BLEU-4: {avg_bleu4:.4f}")

NameError: name 'beam_search_caption_attention' is not defined